# Notebook 04: Repeated 5×5 CV (TF-IDF) — LR and Linear SVM

Runs **Repeated Stratified 5×5 CV** (25 folds) and saves fold scores for:
- Logistic Regression (LR)
- Linear SVM (LinearSVC)

Outputs:
- `outputs/repeated_tfidf_scores_small.txt`
- `outputs/repeated_tfidf_scores_base.txt`
- `outputs/repeated_tfidf_scores_small_svm.txt`
- `outputs/repeated_tfidf_scores_base_svm.txt`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score

PROJECT_ROOT = Path(".").resolve()

def run_repeated_cv(csv_path: Path, model_kind: str, out_scores: Path):
    df = pd.read_csv(csv_path)
    texts = df["text"].values
    y = df["label"].map({"cn": 0, "ad": 1}).values

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=3000,
        stop_words="english",
        min_df=2,
    )
    X = vectorizer.fit_transform(texts)

    n_repeats, n_splits = 5, 5
    scores = []

    for repeat in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + repeat)
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            if model_kind == "lr":
                model = LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=500,
                    solver="liblinear",
                    random_state=42,
                )
            elif model_kind == "svm":
                model = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
            else:
                raise ValueError("model_kind must be 'lr' or 'svm'")

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            bacc = balanced_accuracy_score(y_test, y_pred)
            scores.append(bacc)

    scores = np.array(scores, dtype=float)
    out_scores.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(out_scores, scores)
    return float(scores.mean()), float(scores.std()), scores

base_csv = PROJECT_ROOT/"outputs"/"features"/"all_texts_base.csv"
small_csv = PROJECT_ROOT/"outputs"/"features"/"all_texts_small.csv"

# Logistic Regression
m, s, _ = run_repeated_cv(small_csv, "lr", PROJECT_ROOT/"outputs"/"repeated_tfidf_scores_small.txt")
print("LR + small: mean", round(m,4), "std", round(s,4))

m, s, _ = run_repeated_cv(base_csv, "lr", PROJECT_ROOT/"outputs"/"repeated_tfidf_scores_base.txt")
print("LR + base : mean", round(m,4), "std", round(s,4))

# Linear SVM
m, s, _ = run_repeated_cv(small_csv, "svm", PROJECT_ROOT/"outputs"/"repeated_tfidf_scores_small_svm.txt")
print("SVM + small: mean", round(m,4), "std", round(s,4))

m, s, _ = run_repeated_cv(base_csv, "svm", PROJECT_ROOT/"outputs"/"repeated_tfidf_scores_base_svm.txt")
print("SVM + base : mean", round(m,4), "std", round(s,4))

> If your numbers differ slightly from the terminal scripts, that’s normal if any hyperparameters differ.
Keep seeds and settings consistent for your final paper.